# 06 — Promptfoo Qualitative Characterisation

Uses an LLM-as-judge (Llama-3.3-70b via NVIDIA NIM) to produce a **qualitative reasoning fingerprint**
for each prompt variant evaluated in `05_Prompt_Variance.ipynb`.

Rather than accuracy scores, this answers: *how does each prompt variant reason?*  
What features does it anchor on? Is it conservative or optimistic? Does it reason specifically or generically?

**Prerequisites:** `05_Prompt_Variance.ipynb` must have been run — it produces `data/results/llm/05_reasonings.jsonl`.

---

## Pipeline
| Step | Script | What it does |
|------|--------|--------------|
| 1 | `prepare_tests.py` | Reads JSONL → writes `promptfoo/tests.yaml` (10 samples/variant) |
| 2 | `npx promptfoo eval` | Judge reads each variant's samples, writes qualitative characterisation |
| 3 | This notebook | Loads JSON output, displays characterisations side-by-side |

## Setup

In [1]:
import sys, os, json, subprocess
from pathlib import Path

# Repo root = 3 levels up from this notebook
REPO_ROOT   = Path(os.path.abspath('../../..'))
PROMPTFOO_DIR = REPO_ROOT / 'promptfoo'
RESULTS_DIR   = REPO_ROOT / 'data' / 'results' / 'llm'
JSONL_PATH    = RESULTS_DIR / '05_reasonings.jsonl'
TESTS_YAML    = PROMPTFOO_DIR / 'tests.yaml'
PF_CONFIG     = PROMPTFOO_DIR / 'promptfooconfig.yaml'
PF_OUTPUT     = PROMPTFOO_DIR / 'results' / 'qualitative_characterisations.json'

print(f'Repo root:      {REPO_ROOT}')
print(f'Reasonings file exists: {JSONL_PATH.exists()}')

Repo root:      c:\Users\Jad Zoghaib\OneDrive\Desktop\Sabadell_Capstone
Reasonings file exists: True


---
## Step 1 — Generate `tests.yaml`
Reads 10 reasoning samples per variant (balanced correct/incorrect) from the JSONL.

In [2]:
result = subprocess.run(
    [sys.executable, str(PROMPTFOO_DIR / 'prepare_tests.py')],
    capture_output=True, text=True, cwd=str(REPO_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('prepare_tests.py failed')

print(f'tests.yaml written: {TESTS_YAML.exists()} ({TESTS_YAML.stat().st_size} bytes)')

  [baseline] — 10 samples prepared
  [conservative] — 10 samples prepared
  [chain_of_thought] — 10 samples prepared
  [top_features_only] — 10 samples prepared

Wrote 4 test cases to C:\Users\Jad Zoghaib\OneDrive\Desktop\Sabadell_Capstone\promptfoo\tests.yaml
Next: npx promptfoo@latest eval --config promptfoo/promptfooconfig.yaml

tests.yaml written: True (9870 bytes)


---
## Step 2 — Run Promptfoo Eval
Sends each variant's reasoning samples to the judge. Takes ~1–2 min (5 API calls).

In [ ]:
import dotenv
dotenv.load_dotenv(REPO_ROOT / '.env')

PF_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

cmd = (
    f'npx promptfoo@latest eval'
    f' --config "{PF_CONFIG}"'
    f' --output "{PF_OUTPUT}"'
    f' --no-cache'
)

pf_result = subprocess.run(
    cmd,
    capture_output=True, text=True,
    cwd=str(REPO_ROOT),
    env={**os.environ},
    shell=True,  # required on Windows — npx is a .cmd file
)
print(pf_result.stdout[-3000:] if len(pf_result.stdout) > 3000 else pf_result.stdout)
if pf_result.returncode != 0:
    print('STDERR:', pf_result.stderr[-2000:])
    raise RuntimeError('promptfoo eval failed')

print(f'\nOutput written to: {PF_OUTPUT}')

---
## Step 3 — Display Qualitative Characterisations

In [ ]:
with open(PF_OUTPUT, encoding='utf-8') as f:
    pf_data = json.load(f)

# promptfoo output schema: results.results[]
# Each entry has vars (variant_name, variant_description) and response.output
raw_results = pf_data.get('results', pf_data).get('results', [])

characterisations = []
for entry in raw_results:
    variant  = entry.get('vars', {}).get('variant_name', 'unknown')
    desc     = entry.get('vars', {}).get('variant_description', '')
    output   = entry.get('response', {}).get('output', '') or entry.get('output', '')
    characterisations.append({'variant': variant, 'description': desc, 'characterisation': output})

print(f'Loaded {len(characterisations)} characterisations')
for c in characterisations:
    print(f'  [{c["variant"]}]')

In [ ]:
from IPython.display import display, Markdown

for c in characterisations:
    display(Markdown(
        f"### `{c['variant']}`\n"
        f"*{c['description']}*\n\n"
        f"{c['characterisation']}\n\n---"
    ))

---
## Export — Summary Table

In [ ]:
import pandas as pd

# Merge with Phase 1 quantitative metrics for a combined view
p1_metrics = pd.read_csv(RESULTS_DIR / '05_phase1_metrics.csv')

qual_df = pd.DataFrame(characterisations)[['variant', 'characterisation']]
combined = p1_metrics.merge(qual_df, on='variant', how='left')

cols = ['variant', 'accuracy', 'f1_charged_off', 'n_valid', 'characterisation']
display(combined[cols].style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}))

out_path = RESULTS_DIR / '06_qualitative_summary.csv'
combined[cols].to_csv(out_path, index=False)
print(f'Saved: {out_path}')